# Integration, real-game pilot and launch validation

**Purpose:** validate the complete approved experiment before committing a long run. **Inputs:** model/data/evaluation and exact production-topology recovery receipts. **Outputs:** two source-bound pilot training checkpoints, full real trajectories, equal-initialization checks, `real_pilot.json`, and `delivery_validation.json`. **Runtime:** real pilot roughly 5–15 minutes; validation under one minute. No claim of playing strength comes from this pilot.

**Modes:** default `AHRV3_PREFLIGHT=inspect` imports only; `pilot` runs one real self-play game for B and C, each with eight tactical warmup updates and two replay updates. It interrupts after five real moves and resumes in a fresh kernel. `bundle` archives both completed pilots and verifies fresh-kernel restored inference; `notebooks` executes each component in a fresh kernel and binds the saved receipts to source hashes; `validate` checks all final receipts and source identities. Full training is launched separately only after this passes.

**Checkpoint location:** source-fingerprinted `checkpoints/ahr-v3-pilot-{B,C}-<digest>/`. **Exact resume:** same source/configuration, fresh kernel Run All in pilot mode. Existing valid completed pilots are reused, incomplete ones resume, corrupt manifests or source mismatch fail explicitly. Pilot and full training should run inside tmux for connection independence.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v3-shared:export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb','11_controlled_evaluation.ipynb','13_local_handoff.ipynb','progress.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
_pipeline_namespace={'__name__':'__ahr_v3_pipeline__'}
for dependency in ('notebooks/00_instance_setup_and_recovery.ipynb','reports/09_13_training_pipeline.ipynb'):
    v3_import_exports(PROJECT/dependency,_pipeline_namespace)
# Keep the tested generic controller's implementation and isolated globals.
_v3_original_checkpoint_save=_pipeline_namespace['_save_checkpoint_unlocked']
def _v3_controller_checkpoint_save(*args,**kwargs):
    kwargs['resume_cell']='experiments/ahr_jepa_v3/protocol_and_run.ipynb: AHRV3_MODE=run; fresh kernel Run All'
    return _v3_original_checkpoint_save(*args,**kwargs)
_pipeline_namespace['_save_checkpoint_unlocked']=_v3_controller_checkpoint_save
v3_run_controller=_pipeline_namespace['run_training_pipeline']

v3_import_exports(PROJECT/V3_ROOT/'protocol_and_run.ipynb',globals())


In [ ]:
def v3_run_real_pilot(project):
    project=Path(project);sources=v3_source_identity(project)
    production=v3_read_json(v3_report_path(project,'production_recovery.json'))
    if production.get('status')!='PASS' or production['source_identity']!=sources:raise RuntimeError('Matching production recovery receipt required')
    fingerprint=v3_digest(sources)[:10];runs={};timings={}
    for arm in ('B','C'):
        run=f'ahr-v3-pilot-{arm}-'+fingerprint
        config=V3TrainConfig(seed=2026092799,use_jepa=arm=='C',games=1,warmup_updates=8,updates_per_game=2,batch_size=32,
            device='cuda:0',actor_device='cuda:1',max_plies=256,time_budget_s=1.,reanalysis_per_game=2,
            search={'max_depth':4,'max_nodes':192,'branch_width':8,'queen_cycles':2},purpose='go_training')
        prior=v3_report_path(project,run+'_training.json')
        if prior.exists() and v3_read_json(prior)['status']=='COMPLETE':
            result=v3_read_json(prior);timings[arm]={'status':'REUSED_COMPLETED_PILOT','elapsed_s':None}
        else:
            started=time.perf_counter()
            if not (project/'checkpoints'/run/'manifest.json').exists():
                partial=v3_fresh_training_call(project,run,config,14)
                state,_=v3_read_training_checkpoint(project/'checkpoints'/run)
                if state['phase']!='play' or len(state['current_game']['records'])!=5:raise RuntimeError('Real pilot failed to stop after five actual moves')
            result=v3_fresh_training_call(project,run,config)
            timings[arm]={'status':'MEASURED','elapsed_s':time.perf_counter()-started}
        if result['config']!=asdict(config):raise RuntimeError('Existing pilot configuration differs from declared pilot; use an explicit new run identity')
        if result['status']!='COMPLETE' or result['games_finished']!=1 or result['optimizer_steps']!=10:raise RuntimeError('Real pilot did not complete its declared work')
        if result['metrics']['search_calls']<=0 or result['archive_plies']<=0:raise RuntimeError('Pilot did not use real AHR moves')
        model,metadata=v3_load_model(project,run,device='cpu')
        if not all(torch.isfinite(p).all() for p in model.parameters()):raise RuntimeError('Nonfinite learned pilot model')
        runs[arm]={'run_id':run,'training':result,'checkpoint':metadata}
    if runs['B']['training']['initialization_digest']!=runs['C']['training']['initialization_digest']:raise RuntimeError('B/C pilot initial weights differ')
    configs=[copy.deepcopy(runs[a]['training']['config']) for a in ('B','C')]
    for cfg in configs:cfg.pop('use_jepa')
    if configs[0]!=configs[1]:raise RuntimeError('B/C pilot budgets differ')
    receipt={'status':'PASS','source_identity':sources,'real_ahr_games':2,'training_updates':20,'runs':runs,'timings':timings,
        'same_initial_weights':True,'same_declared_budgets_except_jepa_flag':True,
        'prediction_horizons':[1,2,4,8],'purpose':'Functionality, recovery and timing only; not a strength comparison',
        'cutoffs':'Retained as UNKNOWN in trajectory archive; never outcome supervision'}
    atomic_json(v3_report_path(project,'real_pilot.json'),receipt)
    return receipt

def v3_validate_pilot_bundle(project):
    project=Path(project)
    pilot=v3_read_json(v3_report_path(project,'real_pilot.json'))
    if pilot['status']!='PASS' or pilot['source_identity']!=v3_source_identity(project):
        raise RuntimeError('Current real pilot required before bundle validation')
    runs=[pilot['runs'][arm]['run_id'] for arm in ('B','C')]
    label='pilot-'+v3_digest(v3_all_experiment_sources(project))[:12]
    bundle=v3_export_experiment(project,runs,label=label)
    if bundle['status']!='VERIFIED_LOCAL':raise RuntimeError('Pilot archive restore did not pass')
    receipt={'status':'PASS','sources':v3_all_experiment_sources(project),'run_ids':runs,'bundle':bundle,
             'off_instance_backup':'DEFERRED_BY_USER'}
    atomic_json(v3_report_path(project,'bundle_validation.json'),receipt)
    return receipt

def v3_run_final_notebook_checks(project):
    project=Path(project);before=v3_all_experiment_sources(project)
    names=('00_shared.ipynb','08_predictive_ahr.ipynb','09_trajectory_learning.ipynb',
           '11_controlled_evaluation.ipynb','12_play_predictive_ahr.ipynb','13_local_handoff.ipynb',
           'restore_inference.ipynb','protocol_and_run.ipynb','run_stage.ipynb','progress.ipynb','monitor.ipynb')
    rows=[]
    for name in names:
        path=project/V3_ROOT/name
        env={k:v for k,v in os.environ.items() if not k.startswith('AHR')}
        log=v3_report_path(project,'final-check-'+name.replace('.ipynb','.log'))
        started=time.perf_counter()
        with log.open('w') as output:
            result=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace',
                '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=900',str(path)],
                cwd=project,env=env,stdout=output,stderr=subprocess.STDOUT)
        if result.returncode:raise RuntimeError('Fresh Run All failed: '+str(log))
        book=nbformat.read(path,4);nbformat.validate(book)
        errors=[o for cell in book.cells if cell.cell_type=='code' for o in cell.outputs if o.output_type=='error']
        if errors:raise RuntimeError('Saved notebook contains execution errors: '+name)
        if v3_code_sha(path)!=before[str(path.relative_to(project))]:raise RuntimeError('Source changed during validation: '+name)
        rows.append({'notebook':str(path.relative_to(project)),'status':'PASS','fresh_kernel':True,
                     'code_sha256':v3_code_sha(path),'log':str(log.relative_to(project)),'elapsed_s':time.perf_counter()-started})
        print('Fresh Run All PASS:',name,flush=True)
    if before!=v3_all_experiment_sources(project):raise RuntimeError('Experiment changed while final checks ran')
    receipt={'status':'PASS','sources':before,'notebooks':rows,'training_scope':'Only explicitly labeled bounded fixtures in default validation; full runs not launched'}
    atomic_json(v3_report_path(project,'notebook_execution_validation.json'),receipt)
    return receipt

def v3_validate_delivery(project):
    project=Path(project);sources=v3_source_identity(project)
    v3_verify_original_sources(project)
    execution=v3_read_json(v3_report_path(project,'notebook_execution_validation.json'))
    if execution['status']!='PASS' or execution['sources']!=v3_all_experiment_sources(project):raise RuntimeError('Final fresh-kernel source-bound notebook checks are required')
    receipts={}
    for name in ('model_validation','data_validation','trainer_validation','production_recovery','evaluation_validation','real_pilot','play_validation','trained_play_validation','bundle_validation'):
        path=v3_report_path(project,name+'.json');value=v3_read_json(path)
        if value.get('status')!='PASS':raise RuntimeError('Required validation failed: '+name)
        if name in ('production_recovery','real_pilot','trained_play_validation') and value['source_identity']!=sources:raise RuntimeError('Validation source mismatch: '+name)
        if name=='evaluation_validation' and value['sources']!=sources:raise RuntimeError('Arena validation source mismatch')
        if name=='trained_play_validation' and value['play_code_sha256']!=v3_code_sha(project/V3_ROOT/'12_play_predictive_ahr.ipynb'):raise RuntimeError('Trained UI validation source mismatch')
        if name=='bundle_validation' and value['sources']!=v3_all_experiment_sources(project):raise RuntimeError('Bundle validation source mismatch')
        receipts[name]={'path':str(path.relative_to(project)),'sha256':file_sha256(path)}
    # Every authored notebook is valid and every code cell parses before any long job.
    notebooks=[]
    for path in sorted((project/V3_ROOT).glob('*.ipynb')):
        book=nbformat.read(path,4);nbformat.validate(book)
        for index,cell in enumerate(book.cells):
            if cell.cell_type=='code':compile(cell.source,str(path)+':'+str(index),'exec')
        notebooks.append(str(path.relative_to(project)))
    pilot=v3_read_json(v3_report_path(project,'real_pilot.json'))
    for arm in ('B','C'):
        run=pilot['runs'][arm]['run_id'];model,metadata=v3_load_model(project,run,'cpu')
        # Same fully trained-pilot constructor and frozen-player path that full arenas will use.
        player=v3_frozen_player(project,'candidate',run,arm=arm,device='cpu',queen_cycles=2)
        assert player is not None and metadata['checkpoint_sha256']==pilot['runs'][arm]['checkpoint']['checkpoint_sha256']
    report={'schema':V3_SCHEMA,'status':'PASS','sources':v3_all_experiment_sources(project),
            'core_sources':sources,'original_sources_preserved':True,'production_recovery_passed':True,
            'real_pilot_passed':True,'notebooks':notebooks,'validation_receipts':receipts,
            'off_instance_backup':'DEFERRED_BY_USER','trained_strength':'NOT_ESTABLISHED',
            'validated_at_utc':dt.datetime.now(dt.timezone.utc).isoformat()}
    atomic_json(v3_report_path(project,'delivery_validation.json'),report)
    return report


In [ ]:
mode=os.environ.get('AHRV3_PREFLIGHT','inspect')
if mode=='pilot':
    receipt=v3_run_real_pilot(PROJECT)
    print(json.dumps({'status':receipt['status'],'real_games':receipt['real_ahr_games'],'updates':receipt['training_updates'],'timings':receipt['timings']},indent=2))
elif mode=='bundle':
    receipt=v3_validate_pilot_bundle(PROJECT)
    print(json.dumps({'status':receipt['status'],'runs':receipt['run_ids'],'archive':receipt['bundle']['archive']},indent=2))
elif mode=='notebooks':
    receipt=v3_run_final_notebook_checks(PROJECT)
    print(json.dumps({'status':receipt['status'],'fresh_notebooks':len(receipt['notebooks'])},indent=2))
elif mode=='validate':
    receipt=v3_validate_delivery(PROJECT)
    print(json.dumps({'status':receipt['status'],'notebooks':receipt['notebooks'],'production_recovery_passed':True,'real_pilot_passed':True},indent=2))
elif mode=='inspect':print('INSPECT_ONLY: validation and real-pilot modes are explicit; no training started.')
else:raise ValueError('Choose inspect/pilot/bundle/notebooks/validate')